# 02 — Prepare Labels
Merge ECOSoundSet and InsectSet459, balance classes, and produce the `train.csv` / `val.csv` / `test.csv` splits that OpenSoundscape expects.

**Kernel:** `Python (orthoptera-training)`

In [ ]:
import pandas as pd
import numpy as np
import os

ECO_ROOT    = "../../datasets/ecosoundset"
INSECT_ROOT = "../../datasets/insectset459"
OUT_DIR     = "../../training/data"
os.makedirs(OUT_DIR, exist_ok=True)

UK_SPECIES = [
    'Chorthippus brunneus',
    'Chorthippus parallelus',
    'Omocestus viridulus',
    'Tettigonia viridissima',
    'Roeseliana roeselii',
    'Pholidoptera griseoaptera',
    'Leptophyes punctatissima',
    'Meconema thalassinum',
    'Gryllus campestris',
]

In [ ]:
# ── Load ECOSoundSet strong labels ────────────────────────────────────────────
eco_train = pd.read_csv(os.path.join(ECO_ROOT, "train_strong.csv"))
eco_val   = pd.read_csv(os.path.join(ECO_ROOT, "val_strong.csv"))
eco_test  = pd.read_csv(os.path.join(ECO_ROOT, "test_strong.csv"))

# Add full audio path
for df in [eco_train, eco_val, eco_test]:
    df['filepath'] = df['filename'].apply(
        lambda f: os.path.abspath(os.path.join(ECO_ROOT, "audio", f))
    )
    df['source'] = 'ecosoundset'

# Filter to UK species
eco_train = eco_train[eco_train['species'].isin(UK_SPECIES)]
eco_val   = eco_val[eco_val['species'].isin(UK_SPECIES)]
eco_test  = eco_test[eco_test['species'].isin(UK_SPECIES)]

print(f"ECOSoundSet — train: {len(eco_train)}  val: {len(eco_val)}  test: {len(eco_test)}")

In [ ]:
# ── Optionally supplement with InsectSet459 ───────────────────────────────────
# Only needed for species with < 50 clips in ECOSoundSet.
# Inspect InsectSet459 structure — its column names may differ.

insect_csv = os.path.join(INSECT_ROOT, "metadata.csv")  # adjust filename if different
if os.path.exists(insect_csv):
    insect = pd.read_csv(insect_csv)
    print("InsectSet459 columns:", list(insect.columns))
    print(insect.head())
else:
    print("InsectSet459 not downloaded yet — skipping supplement step.")
    print("Run: cd datasets/insectset459 && zenodo_get 14056458")

In [ ]:
# ── Build OpenSoundscape one-hot label CSVs ───────────────────────────────────
# OSS expects: file | start_time | end_time | Species1 | Species2 | ...
# (one-hot encoded, 1 = present in that clip segment)

def to_oss_format(df, species_list):
    rows = []
    for _, row in df.iterrows():
        entry = {
            'file':       row['filepath'],
            'start_time': row.get('t_min', 0.0),
            'end_time':   row.get('t_max', 3.0),
        }
        for sp in species_list:
            entry[sp] = 1 if row['species'] == sp else 0
        rows.append(entry)
    return pd.DataFrame(rows).set_index(['file', 'start_time', 'end_time'])

train_oss = to_oss_format(eco_train, UK_SPECIES)
val_oss   = to_oss_format(eco_val,   UK_SPECIES)
test_oss  = to_oss_format(eco_test,  UK_SPECIES)

train_oss.to_csv(os.path.join(OUT_DIR, "train.csv"))
val_oss.to_csv(os.path.join(OUT_DIR,   "val.csv"))
test_oss.to_csv(os.path.join(OUT_DIR,  "test.csv"))

print(f"Saved to {OUT_DIR}/")
print(f"Train: {len(train_oss)}  Val: {len(val_oss)}  Test: {len(test_oss)}")
train_oss.sum().sort_values(ascending=False)